In [ ]:
# Import required libraries

from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from tensorflow.keras import layers, Model, Sequential
from tensorflow.keras import optimizers, losses, callbacks

In [ ]:
# set seeds for reproducibility
seed = 42

os.environ['PYTHONHASHSEED'] = str(seed)

random.seed(seed)
np.random.seed(seed)

tf.random.set_seed(seed)

# Force TensorFlow to use deterministic operations
# This may slightly reduce performance but ensures consistency
tf.config.experimental.enable_op_determinism()

In [ ]:
# Helper function to save model results

def save_model_results(model_name, history, y_true, y_pred):
    
    hist_df = pd.DataFrame(history.history)
    hist_df.to_csv(f'{model_name}_history.csv', index=False)
    
    
    results_df = pd.DataFrame({
        'True_Values': y_true.flatten(),
        'Predictions': y_pred.flatten()
    })
    results_df.to_csv(f'{model_name}_predictions.csv', index=False)
    
    print(f"✅ Saved results for {model_name} to CSV.")

Do not import tensorflow and pytorch in the same notebook as it crashes the notebook

# Dataset 1: Bike Sharing

### Characteristics

In [9]:
# fetch dataset 
bike_sharing = fetch_ucirepo(id=275) 
  
# data (as pandas dataframes) 
X_bike = bike_sharing.data.features 
y_bike = bike_sharing.data.targets 

bike_df = pd.concat([X_bike, y_bike], axis=1)


Characteristic 1 & 2.

In [10]:
num_variables = bike_df.shape[1]
num_records = bike_df.shape[0]
print(f"Number of variables (columns): {num_variables}")
print(f"Number of records (rows): {num_records}\n")

Number of variables (columns): 14
Number of records (rows): 17379



Characteristic 3.

In [11]:
bike_df.head(10)

,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,cnt
0,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0000,16
1,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0000,40
2,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0000,32
3,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0000,13
4,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0000,1
5,2011-01-01,1,0,1,5,0,6,0,2,0.24,0.2576,0.75,0.0896,1
6,2011-01-01,1,0,1,6,0,6,0,1,0.22,0.2727,0.80,0.0000,2
7,2011-01-01,1,0,1,7,0,6,0,1,0.20,0.2576,0.86,0.0000,3
8,2011-01-01,1,0,1,8,0,6,0,1,0.24,0.2879,0.75,0.0000,8
9,2011-01-01,1,0,1,9,0,6,0,1,0.32,0.3485,0.76,0.0000,14


In [12]:
bike_df.tail(10)

,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,cnt
17369,2012-12-31,1,1,12,14,0,1,1,2,0.28,0.2727,0.45,0.2239,247
17370,2012-12-31,1,1,12,15,0,1,1,2,0.28,0.2879,0.45,0.1343,315
17371,2012-12-31,1,1,12,16,0,1,1,2,0.26,0.2576,0.48,0.1940,214
17372,2012-12-31,1,1,12,17,0,1,1,2,0.26,0.2879,0.48,0.0896,164
17373,2012-12-31,1,1,12,18,0,1,1,2,0.26,0.2727,0.48,0.1343,122
17374,2012-12-31,1,1,12,19,0,1,1,2,0.26,0.2576,0.60,0.1642,119
17375,2012-12-31,1,1,12,20,0,1,1,2,0.26,0.2576,0.60,0.1642,89
17376,2012-12-31,1,1,12,21,0,1,1,1,0.26,0.2576,0.60,0.1642,90
17377,2012-12-31,1,1,12,22,0,1,1,1,0.26,0.2727,0.56,0.1343,61
17378,2012-12-31,1,1,12,23,0,1,1,1,0.26,0.2727,0.65,0.1343,49


In [13]:
print("Data types of each column:")
print(bike_df.dtypes)
print("\n")

Data types of each column:
dteday         object
season          int64
yr              int64
mnth            int64
hr              int64
holiday         int64
weekday         int64
workingday      int64
weathersit      int64
temp          float64
atemp         float64
hum           float64
windspeed     float64
cnt             int64
dtype: object




| Column      | Type         | Subtype            | Explanation |
|-------------|--------------|--------------------|-------------|
| index       | Categorical  | Nominal            | Row identifier with no numeric meaning |
| dteday      | Categorical  | Nominal (date)     | Treated as a label unless decomposed |
| season      | Categorical  | Ordinal            | Encoded seasons with natural order (1–4) |
| yr          | Categorical  | Ordinal            | 0 = 2011, 1 = 2012 (progression in time) |
| mnth        | Categorical  | Ordinal            | Months have natural order (1–12) |
| hr          | Categorical  | Ordinal            | Hours of day (0–23) with natural order |
| holiday     | Categorical  | Binary Nominal     | Holiday flag (0/1) |
| weekday     | Categorical  | Nominal            | Days of week (0–6), no natural order |
| workingday  | Categorical  | Binary Nominal     | Working day flag (0/1) |
| weathersit  | Categorical  | Ordinal            | Weather categories ranked by severity (1–4) |
| temp        | Numerical    | Continuous         | Normalized temperature |
| atemp       | Numerical    | Continuous         | Normalized “feels‑like” temperature |
| hum         | Numerical    | Continuous         | Normalized humidity |
| windspeed   | Numerical    | Continuous         | Normalized windspeed |
| cnt         | Numerical    | Discrete           | Count of total bike rentals |


Characteristic 4.

In [14]:

# Generate descriptive (summary) statistics for bike_df.
print("Descriptive (summary) statistics:")
print(bike_df.describe())
print("\n")

Descriptive (summary) statistics:
             season            yr          mnth            hr       holiday  \
count  17379.000000  17379.000000  17379.000000  17379.000000  17379.000000   
mean       2.501640      0.502561      6.537775     11.546752      0.028770   
std        1.106918      0.500008      3.438776      6.914405      0.167165   
min        1.000000      0.000000      1.000000      0.000000      0.000000   
25%        2.000000      0.000000      4.000000      6.000000      0.000000   
50%        3.000000      1.000000      7.000000     12.000000      0.000000   
75%        3.000000      1.000000     10.000000     18.000000      0.000000   
max        4.000000      1.000000     12.000000     23.000000      1.000000   

            weekday    workingday    weathersit          temp         atemp  \
count  17379.000000  17379.000000  17379.000000  17379.000000  17379.000000   
mean       3.003683      0.682721      1.425283      0.496987      0.475775   
std        2.0057

Characteristics 5

In [15]:
# Check for and print the total count of missing values for each column in bike_df.
missing_values = bike_df.isnull().sum()
print("Missing values per column:")
print(missing_values[missing_values > 0])
if missing_values.sum() == 0:
    print("No missing values found.\n")
else:
    print("\n")

# Check for and print the total count of duplicate rows in bike_df.
duplicate_rows = bike_df.duplicated().sum()
print(f"Total count of duplicate rows: {duplicate_rows}\n")

# Print the column names of bike_df to identify potentially irrelevant columns.
print("Column names in bike_df:")
print(bike_df.columns.tolist())
print("\n")


Missing values per column:
Series([], dtype: int64)
No missing values found.

Total count of duplicate rows: 0

Column names in bike_df:
['dteday', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'cnt']




Characteristics 6.

Balancing assessment: The target variable 'cnt' is a continuous numerical variable representing bike rental counts. This indicates a regression problem, not a classification problem. Therefore, dataset balancing (e.g., handling class imbalance) is not relevant for this task.

In [16]:
# this columns where identified thanks to the summary statistics

numerical_features = ['temp', 'atemp', 'hum', 'windspeed']
ordinal_features = ['hr', 'mnth', 'season', 'weathersit']
binary_features = ['holiday', 'workingday']  # do NOT scale
target = ['cnt']  # regression target

# Combine features to scale
features_to_scale = numerical_features + ordinal_features

scaler = MinMaxScaler()

bike_scaled_df = bike_df.copy()
bike_scaled_df[features_to_scale] = scaler.fit_transform(bike_df[features_to_scale])

bike_scaled_df['cnt'] = bike_df['cnt']

bike_scaled_df.head()


,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,cnt
0,2011-01-01,0.0,0,0.0,0.000000,0,6,0,0.0,0.224490,0.2879,0.81,0.0,16
1,2011-01-01,0.0,0,0.0,0.043478,0,6,0,0.0,0.204082,0.2727,0.80,0.0,40
2,2011-01-01,0.0,0,0.0,0.086957,0,6,0,0.0,0.204082,0.2727,0.80,0.0,32
3,2011-01-01,0.0,0,0.0,0.130435,0,6,0,0.0,0.224490,0.2879,0.75,0.0,13
4,2011-01-01,0.0,0,0.0,0.173913,0,6,0,0.0,0.224490,0.2879,0.75,0.0,1


Characteristic 7.

In [17]:
train_size = int(0.70 * num_records)
test_size = num_records - train_size
print(f"If a 70/30 split were performed:")
print(f"  Training set records (70%): {train_size}")
print(f"  Test set records (30%): {test_size}")

If a 70/30 split were performed:
  Training set records (70%): 12165
  Test set records (30%): 5214


### Dataset preprocessing:
* Convert to supervised learning (sliding window)
* Create sequences (e.g., 24‑hour window → next hour prediction)

In [ ]:
# Convert scaled bike dataframe to numpy array
bike_scaled_df.drop(columns=['dteday'], inplace=True)

# save dataset before creating sequences as the sequence make it 3d
pd.to_csv(bike_scaled_df, 'bike_scaled_data.csv', index=False)

bike_data = bike_scaled_df.values

lookback_window = 24

# Create sequences using sliding window
X_bike_seq, y_bike_seq = [], []

for i in range(len(bike_data) - lookback_window):
    X_bike_seq.append(bike_data[i:i + lookback_window, :-1])
    # Target: next hour's bike count (cnt)
    y_bike_seq.append(bike_data[i + lookback_window, -1])

X_bike_seq = np.array(X_bike_seq)
y_bike_seq = np.array(y_bike_seq)

print(f"Sequence shape - X: {X_bike_seq.shape}, y: {y_bike_seq.shape}")
print(f"Number of sequences: {X_bike_seq.shape[0]}")
print(f"Time steps per sequence: {X_bike_seq.shape[1]}")
print(f"Number of features: {X_bike_seq.shape[2]}\n")


Sequence shape - X: (17355, 24, 12), y: (17355,)
Number of sequences: 17355
Time steps per sequence: 24
Number of features: 12



The dteday column is dropped because it contains string-formatted timestamps which are incompatible with the mathematical operations in PyTorch and TensorFlow. This does not affect the model's performance because the temporal information is already preserved through the numeric features yr, mnth, hr, holiday, weekday, and workingday. Dropping the raw string ensures the dataset is a purely numeric $float32$ matrix, which is required for tensor conversion.

In [19]:
# Split into train and test sets (70/30 split)
train_size = int(0.70 * len(X_bike_seq))
test_size = len(X_bike_seq) - train_size

X_bike_train = X_bike_seq[:train_size]
y_bike_train = y_bike_seq[:train_size]

X_bike_test = X_bike_seq[train_size:]
y_bike_test = y_bike_seq[train_size:]

print(f"Training set:")
print(f"  X shape: {X_bike_train.shape}, y shape: {y_bike_train.shape}")
print(f"Training samples: {X_bike_train.shape[0]}")
print(f"\nTest set:")
print(f"  X shape: {X_bike_test.shape}, y shape: {y_bike_test.shape}")
print(f"Test samples: {X_bike_test.shape[0]}\n")

Training set:
  X shape: (12148, 24, 12), y shape: (12148,)
Training samples: 12148

Test set:
  X shape: (5207, 24, 12), y shape: (5207,)
Test samples: 5207



In [ ]:
# for tensor flow implementations:
bike_train_dataset = tf.data.Dataset.from_tensor_slices((X_bike_train, y_bike_train))
bike_test_dataset = tf.data.Dataset.from_tensor_slices((X_bike_test, y_bike_test))

batch_size = 32
bike_train_dataset = bike_train_dataset.batch(batch_size)
bike_test_dataset = bike_test_dataset.batch(batch_size)

print("TensorFlow datasets created:")

TensorFlow datasets created:


2026-03-01 13:31:00.450525: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-03-01 13:31:00.450570: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 32.00 GB
2026-03-01 13:31:00.450576: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 12.48 GB
2026-03-01 13:31:00.450606: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-03-01 13:31:00.450615: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


## Technique 1: Vanilla RNN

In [ ]:

class VanillaRNNTF(Model):
    def __init__(self, input_size, hidden_size, output_size, num_layers=2):
        super(VanillaRNNTF, self).__init__()
        
        # We wrap the LSTM in a Bidirectional layer.
        # To mimic num_layers, we stack them or use a loop if num_layers > 1.
        # For num_layers=2, we use a StackedRNNCell or sequential LSTMs.
        
        self.lstm_layers = []
        for i in range(num_layers):
            self.lstm_layers.append(
                layers.Bidirectional(
                    layers.LSTM(
                        hidden_size, 
                        return_sequences=True if i < num_layers - 1 else False,
                        return_state=True,
                        dropout=0.3
                    )
                )
            )

        # The Fully Connected (fc) block
        self.fc = tf.keras.Sequential([
            layers.Dense(64, activation='relu'),
            layers.Dropout(0.2),
            layers.Dense(output_size)
        ])

    def call(self, x):
        # In TF: The Bidirectional(return_state=True) returns [output, forward_h, forward_c, backward_h, backward_c]
        
        current_input = x
        final_state = None
        
        for i, lstm in enumerate(self.lstm_layers):
            if i == len(self.lstm_layers) - 1:
                # For the last layer, we capture the final states
                output, forward_h, forward_c, backward_h, backward_c = lstm(current_input)
                # Concatenate forward and backward hidden states to match PyTorch logic
                final_state = tf.concat([forward_h, backward_h], axis=-1)
            else:
                # Intermediate layers just pass sequences forward
                current_input, _, _, _, _ = lstm(current_input)
        
        return self.fc(final_state)


In [ ]:


hidden_size = 128
lr = 0.0005
model = VanillaRNNTF(input_size=12, hidden_size=hidden_size, output_size=1)

optimizer = optimizers.AdamW(
    learning_rate=lr, 
    weight_decay=1e-4, 
    global_clipnorm=1.0
)

model.compile(
    optimizer=optimizer,
    loss=losses.Huber(), # Robust to outliers
)

# Scheduling and Early Stopping
callbacks_list = [
    # Matches: optim.lr_scheduler.ReduceLROnPlateau(patience=5, factor=0.5)
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', 
        factor=0.5, 
        patience=5, 
        verbose=1
    ),
    # Matches: Early Stopping logic (patience=15)
    callbacks.EarlyStopping(
        monitor='val_loss', 
        patience=15, 
        restore_best_weights=True, 
        verbose=1
    ),
    # Matches: torch.save(model.state_dict(), 'best_rnn_model.pth')
    callbacks.ModelCheckpoint(
        filepath='best_rnn_model.weights.h5',
        monitor='val_loss',
        save_best_only=True,
        save_weights_only=True
    )

]

# 5. Training Loop
history = model.fit(
    bike_train_dataset,
    epochs=150,
    validation_data=bike_test_dataset,
    callbacks=callbacks_list,
    verbose=1 
)


Epoch 1/150


2026-03-01 13:31:01.558736: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


380/380 ━━━━━━━━━━━━━━━━━━━━ 31s 70ms/step - loss: 88.2778 - val_loss: 173.3054 - learning_rate: 5.0000e-04
Epoch 2/150
380/380 ━━━━━━━━━━━━━━━━━━━━ 26s 69ms/step - loss: 85.2829 - val_loss: 134.1346 - learning_rate: 5.0000e-04
Epoch 3/150
380/380 ━━━━━━━━━━━━━━━━━━━━ 27s 70ms/step - loss: 71.2880 - val_loss: 122.0515 - learning_rate: 5.0000e-04
Epoch 4/150
380/380 ━━━━━━━━━━━━━━━━━━━━ 27s 70ms/step - loss: 70.5167 - val_loss: 123.1350 - learning_rate: 5.0000e-04
Epoch 5/150
380/380 ━━━━━━━━━━━━━━━━━━━━ 27s 70ms/step - loss: 69.6733 - val_loss: 111.8943 - learning_rate: 5.0000e-04
Epoch 6/150
380/380 ━━━━━━━━━━━━━━━━━━━━ 27s 71ms/step - loss: 66.2185 - val_loss: 100.6625 - learning_rate: 5.0000e-04
Epoch 7/150
380/380 ━━━━━━━━━━━━━━━━━━━━ 27s 71ms/step - loss: 65.5166 - val_loss: 94.9473 - learning_rate: 5.0000e-04
Epoch 8/150
380/380 ━━━━━━━━━━━━━━━━━━━━ 27s 71ms/step - loss: 56.7495 - val_loss: 90.8923 - learning_rate: 5.0000e-04
Epoch 9/150
380/380 ━━━━━━━━━━━━━━━━━━━━ 29s 75ms/step

In [ ]:
# Generate predictions for the CSV
y_pred = model.predict(bike_test_dataset)
y_true = np.concatenate([y for x, y in bike_test_dataset], axis=0)

# SAVE EVERYTHING
save_model_results('Vanilla_RNN_bike_tf', history, y_true, y_pred)

In [ ]:

model.load_weights('best_rnn_model.weights.h5')

y_pred = model.predict(bike_test_dataset).flatten()

# Extract True Labels from the tf.data.Dataset
# In TF, we need to iterate once to get the y values into a single numpy array
y_true = np.concatenate([y for x, y in bike_test_dataset], axis=0)

# Calculate and Print Metrics
print(f"\n--- TensorFlow Improved Metrics ---")
print(f"R² Score: {r2_score(y_true, y_pred):.4f}")
print(f"MSE:       {mean_squared_error(y_true, y_pred):.4f}")
print(f"RMSE:      {np.sqrt(mean_squared_error(y_true, y_pred)):.4f}")
print(f"MAE:       {mean_absolute_error(y_true, y_pred):.4f}")

163/163 ━━━━━━━━━━━━━━━━━━━━ 4s 25ms/step

--- TensorFlow Improved Metrics ---
R² Score: 0.8675
MSE:       6435.0561
RMSE:      80.2188
MAE:       51.9605


2026-03-01 13:53:39.741015: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## Technique 2: RestNet

In [ ]:
class ResBlock1D(layers.Layer):
    def __init__(self, channels, dropout_rate=0.2, **kwargs):
        super(ResBlock1D, self).__init__(**kwargs)
        # padding='same' in TF is equivalent to padding=1 with kernel=3 in PyTorch
        self.conv1 = layers.Conv1D(channels, kernel_size=3, padding='same')
        self.bn1 = layers.BatchNormalization()
        self.relu = layers.LeakyReLU(alpha=0.1)
        self.dropout = layers.Dropout(dropout_rate)
        self.conv2 = layers.Conv1D(channels, kernel_size=3, padding='same')
        self.bn2 = layers.BatchNormalization()

    def call(self, x, training=False):
        residual = x
        out = self.conv1(x)
        out = self.bn1(out, training=training)
        out = self.relu(out)
        out = self.dropout(out, training=training)
        
        out = self.conv2(out)
        out = self.bn2(out, training=training)
        
        # Residual connection + final activation
        return self.relu(out + residual)

class ImprovedResNet1D(Model):
    def __init__(self, hidden_channels=64, **kwargs):
        super(ImprovedResNet1D, self).__init__(**kwargs)
        # PyTorch padding=2 for kernel=5 is 'same' in TF
        self.init_conv = layers.Conv1D(hidden_channels, kernel_size=5, padding='same')
        
        self.layer1 = ResBlock1D(hidden_channels)
        self.layer2 = ResBlock1D(hidden_channels)
        
        # GlobalAveragePooling1D matches AdaptiveAvgPool1d(1) + squeeze
        self.gap = layers.GlobalAveragePooling1D()
        
        self.fc = Sequential([
            layers.Dense(32, activation='relu'),
            layers.Dense(1)
        ])

    def call(self, x, training=False):
        # Note: No transpose needed. TF expects (Batch, Seq, Features)
        x = self.init_conv(x)
        x = tf.nn.relu(x)
        x = self.layer1(x, training=training)
        x = self.layer2(x, training=training)
        x = self.gap(x)
        return self.fc(x)

In [ ]:
# Initialize
model_res = ImprovedResNet1D(hidden_channels=64)

# AdamW with specific weight decay
optimizer = tf.keras.optimizers.AdamW(learning_rate=0.001, weight_decay=1e-2)

model_res.compile(optimizer=optimizer, loss=tf.keras.losses.Huber())

resnet_callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=7, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(filepath='best_resnet.weights.h5', save_best_only=True, save_weights_only=True)
]

# Train
history = model_res.fit(
    bike_train_dataset,
    epochs=150,
    validation_data=bike_test_dataset,
    callbacks=resnet_callbacks
)

Epoch 1/150


380/380 ━━━━━━━━━━━━━━━━━━━━ 13s 24ms/step - loss: 78.4570 - val_loss: 121.5289 - learning_rate: 0.0010
Epoch 2/150
380/380 ━━━━━━━━━━━━━━━━━━━━ 9s 22ms/step - loss: 67.5437 - val_loss: 264.2022 - learning_rate: 0.0010
Epoch 3/150
380/380 ━━━━━━━━━━━━━━━━━━━━ 9s 22ms/step - loss: 61.3329 - val_loss: 525.7027 - learning_rate: 0.0010
Epoch 4/150
380/380 ━━━━━━━━━━━━━━━━━━━━ 9s 23ms/step - loss: 59.2826 - val_loss: 118.2908 - learning_rate: 0.0010
Epoch 5/150
380/380 ━━━━━━━━━━━━━━━━━━━━ 9s 23ms/step - loss: 54.9025 - val_loss: 125.4093 - learning_rate: 0.0010
Epoch 6/150
380/380 ━━━━━━━━━━━━━━━━━━━━ 9s 23ms/step - loss: 51.7610 - val_loss: 118.1099 - learning_rate: 0.0010
Epoch 7/150
380/380 ━━━━━━━━━━━━━━━━━━━━ 9s 23ms/step - loss: 50.7570 - val_loss: 153.5538 - learning_rate: 0.0010
Epoch 8/150
380/380 ━━━━━━━━━━━━━━━━━━━━ 9s 23ms/step - loss: 48.5402 - val_loss: 138.1204 - learning_rate: 0.0010
Epoch 9/150
380/380 ━━━━━━━━━━━━━━━━━━━━ 9s 23ms/step - loss: 46.5457 - val_loss: 143.2289 

In [ ]:
# Generate predictions for the CSV
y_pred = model_res.predict(bike_test_dataset)
y_true = np.concatenate([y for x, y in bike_test_dataset], axis=0)

# SAVE EVERYTHING
save_model_results('RestNet_bike_tf', history, y_true, y_pred)

In [30]:
# Load weights
model_res.load_weights('best_resnet.weights.h5')

# Predict
y_pred = model_res.predict(bike_test_dataset).flatten()
y_true = np.concatenate([y for x, y in bike_test_dataset], axis=0)

print("\n--- TensorFlow Improved 1D-CNN ResNet Metrics ---")
print(f"R² Score: {r2_score(y_true, y_pred):.4f}")
print(f"MSE:       {mean_squared_error(y_true, y_pred):.4f}")
print(f"RMSE:      {np.sqrt(mean_squared_error(y_true, y_pred)):.4f}")
print(f"MAE:       {mean_absolute_error(y_true, y_pred):.4f}")

163/163 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

--- TensorFlow Improved 1D-CNN ResNet Metrics ---
R² Score: 0.4669
MSE:       25893.7548
RMSE:      160.9154
MAE:       118.6092


2026-03-01 14:03:46.810956: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


# Dataset 2: Individual Household Electric Power Consumption

### Characteristics

In [31]:
# fetch dataset
individual_household_electric_power_consumption = fetch_ucirepo(id=235)

# data (as pandas dataframes)
electric_X = individual_household_electric_power_consumption.data.features
electric_y = individual_household_electric_power_consumption.data.targets
electric_df = pd.concat([electric_X, electric_y], axis=1)

/opt/miniconda3/envs/tf_macos/lib/python3.9/site-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (2,3,4,5,6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


Characteristic 1 & 2.

In [32]:
num_variables = electric_df.shape[1]
num_records = electric_df.shape[0]
print(f"Number of variables (columns): {num_variables}")
print(f"Number of records (rows): {num_records}\n")

Number of variables (columns): 9
Number of records (rows): 2075259



Characteristic 3.

In [33]:
electric_df.head(10)

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0
5,16/12/2006,17:29:00,3.520,0.522,235.020,15.000,0.000,2.000,17.0
6,16/12/2006,17:30:00,3.702,0.520,235.090,15.800,0.000,1.000,17.0
7,16/12/2006,17:31:00,3.700,0.520,235.220,15.800,0.000,1.000,17.0
8,16/12/2006,17:32:00,3.668,0.510,233.990,15.800,0.000,1.000,17.0
9,16/12/2006,17:33:00,3.662,0.510,233.860,15.800,0.000,2.000,16.0


In [34]:
print("Data types of each column:")
print(electric_df.dtypes)
print("\n")

Data types of each column:
Date                      object
Time                      object
Global_active_power       object
Global_reactive_power     object
Voltage                   object
Global_intensity          object
Sub_metering_1            object
Sub_metering_2            object
Sub_metering_3           float64
dtype: object




some columns are set to objects, rather than a numerical type, due to this they have to be parse to they can be taken into account as the type of data they actually are

In [35]:
numeric_cols = [
    'Global_active_power',
    'Global_reactive_power',
    'Voltage',
    'Global_intensity',
    'Sub_metering_1',
    'Sub_metering_2',
    'Sub_metering_3'
]
electric_df[numeric_cols] = electric_df[numeric_cols].apply(
    pd.to_numeric,
    errors='coerce'  # invalid values (e.g. '?') become NaN
)

| Column                | Type        | Subtype        | Explanation |
|-----------------------|-------------|----------------|-------------|
| Date                  | Categorical | Nominal        | Calendar date label, no numeric meaning |
| Time                  | Categorical | Nominal        | Time-of-day label, no numeric meaning |
| Global_active_power   | Numerical   | Continuous     | Real-valued household active power consumption (kW) |
| Global_reactive_power | Numerical   | Continuous     | Real-valued reactive power consumption (kW) |
| Voltage               | Numerical   | Continuous     | Real-valued voltage measurement (volts) |
| Global_intensity      | Numerical   | Continuous     | Real-valued current intensity (amps) |
| Sub_metering_1        | Numerical   | Discrete       | Integer energy consumption count for kitchen appliances |
| Sub_metering_2        | Numerical   | Discrete       | Integer energy consumption count for laundry appliances |
| Sub_metering_3        | Numerical   | Discrete       | Integer energy consumption count for water heater & AC |


Characteristic 4.

In [36]:

# Generate descriptive (summary) statistics for electric_df.
print("Descriptive (summary) statistics:")
print(electric_df.describe())
print("\n")

Descriptive (summary) statistics:
       Global_active_power  Global_reactive_power       Voltage  \
count         2.049280e+06           2.049280e+06  2.049280e+06   
mean          1.091615e+00           1.237145e-01  2.408399e+02   
std           1.057294e+00           1.127220e-01  3.239987e+00   
min           7.600000e-02           0.000000e+00  2.232000e+02   
25%           3.080000e-01           4.800000e-02  2.389900e+02   
50%           6.020000e-01           1.000000e-01  2.410100e+02   
75%           1.528000e+00           1.940000e-01  2.428900e+02   
max           1.112200e+01           1.390000e+00  2.541500e+02   

       Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  
count      2.049280e+06    2.049280e+06    2.049280e+06    2.049280e+06  
mean       4.627759e+00    1.121923e+00    1.298520e+00    6.458447e+00  
std        4.444396e+00    6.153031e+00    5.822026e+00    8.437154e+00  
min        2.000000e-01    0.000000e+00    0.000000e+00    0.00000

Characteristics 5

In [37]:
# Check for and print the total count of missing values for each column in electric_df.
missing_values = electric_df.isnull().sum()
print("Missing values per column:")
print(missing_values[missing_values > 0])
if missing_values.sum() == 0:
    print("No missing values found.\n")
else:
    print("\n")

# Check for and print the total count of duplicate rows in electric_df.
duplicate_rows = electric_df.duplicated().sum()
print(f"Total count of duplicate rows: {duplicate_rows}\n")

# Print the column names of electric_df to identify potentially irrelevant columns.
print("Column names in electric_df:")
print(electric_df.columns.tolist())
print("\n")


Missing values per column:
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64


Total count of duplicate rows: 0

Column names in electric_df:
['Date', 'Time', 'Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']




Characteristics 6.

Balancing assessment: The target variable 'Global_active_power' is a continuous numerical variable representing household power consumption in kilowatts. This indicates a regression problem, not a classification problem. Therefore, dataset balancing (e.g., handling class imbalance) is not relevant for this task.

In [38]:
# this columns where identified thanks to the summary statistics

numeric_features = [ 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3' ]


scaler = MinMaxScaler()

electric_scaled_df = electric_df.copy()
electric_scaled_df[numeric_features] = scaler.fit_transform(electric_df[numeric_features])

electric_scaled_df['Global_active_power'] = electric_df['Global_active_power']

electric_scaled_df.head()


,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.300719,0.376090,0.377593,0.0,0.0125,0.548387
1,16/12/2006,17:25:00,5.360,0.313669,0.336995,0.473029,0.0,0.0125,0.516129
2,16/12/2006,17:26:00,5.374,0.358273,0.326010,0.473029,0.0,0.0250,0.548387
3,16/12/2006,17:27:00,5.388,0.361151,0.340549,0.473029,0.0,0.0125,0.548387
4,16/12/2006,17:28:00,3.666,0.379856,0.403231,0.323651,0.0,0.0125,0.548387


Characteristic 7.

In [39]:
train_size = int(0.70 * num_records)
test_size = num_records - train_size
print(f"If a 70/30 split were performed:")
print(f"  Training set records (70%): {train_size}")
print(f"  Test set records (30%): {test_size}")

If a 70/30 split were performed:
  Training set records (70%): 1452681
  Test set records (30%): 622578


### Dataset preprocessing:
* Resample to hourly
* Create sequences

note: I will be not as verbose as in the previous dataset as since most check and justification still hold on this one

In [ ]:
datetime_str = electric_scaled_df['Date'] + ' ' + electric_scaled_df['Time']
electric_scaled_df['DateTime'] = pd.to_datetime(datetime_str, dayfirst=True)

electric_scaled_df.set_index('DateTime', inplace=True)
df_numeric = electric_scaled_df.select_dtypes(include=[np.number])

# Using mean() to aggregate the minutes into an hour
df_hourly = df_numeric.resample('H').mean()

# Handle Potential Missing Values (Common after resampling)
df_hourly = df_hourly.ffill() 

# Feature Engineering: Replicate the 'Time Context' 
df_hourly['hour'] = df_hourly.index.hour
df_hourly['day_of_week'] = df_hourly.index.dayofweek
df_hourly['month'] = df_hourly.index.month

# save dataset before creating sequences as the sequence make it 3d
pd.to_csv(df_hourly, 'electric_hourly_data.csv', index=False)

# Statement: We drop the DateTime index here as yr/mnth/hr features now represent time.
power_data = df_hourly.values.astype(np.float32)

# Create Sequences (Sliding Window)
lookback_window = 24 # 24 hours
X_list, y_list = [], []

for i in range(len(power_data) - lookback_window):
    X_list.append(power_data[i : i + lookback_window, 1:]) 
    y_list.append(power_data[i + lookback_window, 0])

X_power_seq = np.array(X_list)
y_power_seq = np.array(y_list)

/var/folders/s7/06176ck52670b55rv_lly5980000gn/T/ipykernel_21302/2462929361.py:8: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df_hourly = df_numeric.resample('H').mean()


In [41]:
# Split and Convert
train_size = int(0.70 * len(X_power_seq))

print(f"Resampled Shape: {df_hourly.shape}")
print(f"Sequence Shape: {X_power_seq.shape}")

Resampled Shape: (34589, 10)
Sequence Shape: (34565, 24, 9)


In [43]:
# TensorFlow
train_ds_tf = tf.data.Dataset.from_tensor_slices(
    (X_power_seq[:train_size], y_power_seq[:train_size])
).batch(32)
test_ds_tf = tf.data.Dataset.from_tensor_slices(
    (X_power_seq[train_size:], y_power_seq[train_size:])
).batch(32)

## Technique 1: Vanilla RNN

In [ ]:


class ImprovedPowerRNNTF(Model):
    def __init__(self, hidden_size, output_size, num_layers=2, **kwargs):
        super(ImprovedPowerRNNTF, self).__init__(**kwargs)
        
        # return_sequences=True is needed for all but the last layer.
        self.gru_layers = Sequential()
        for i in range(num_layers):
            self.gru_layers.add(
                layers.GRU(
                    hidden_size,
                    return_sequences=True if i < num_layers - 1 else True, # We need sequences for the last layer to grab [:, -1, :]
                    dropout=0.2,
                    recurrent_dropout=0  # Keeping it 0 to use cuDNN kernels for speed
                )
            )
        
        # LayerNorm matches nn.LayerNorm
        self.ln = layers.LayerNormalization()
        
        self.fc = Sequential([
            layers.Dense(32),
            layers.LeakyReLU(alpha=0.1),
            layers.Dense(output_size)
        ])

    def call(self, x, training=False):
        out = self.gru_layers(x, training=training)
        
        # Take the last time step: out[:, -1, :]
        last_step = out[:, -1, :]
        
        # Normalize and pass through FC
        normalized = self.ln(last_step)
        return self.fc(normalized)

In [46]:
# Initialize
model_power = ImprovedPowerRNNTF(hidden_size=128, output_size=1, num_layers=2)

# Optimizer & Loss
# Note: Weight decay 1e-3
optimizer = tf.keras.optimizers.AdamW(learning_rate=0.001, weight_decay=1e-3)
model_power.compile(optimizer=optimizer, loss=tf.keras.losses.Huber())

# Callbacks
power_callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', mode='min', patience=5, factor=0.5, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(filepath='best_power_RNN..weights.h5', save_best_only=True, save_weights_only=True)
]

# Train
history = model_power.fit(
    train_ds_tf,
    epochs=50,
    validation_data=test_ds_tf,
    callbacks=power_callbacks
)

Epoch 1/50
757/757 ━━━━━━━━━━━━━━━━━━━━ 16s 19ms/step - loss: 0.4192 - val_loss: 0.2417 - learning_rate: 0.0010
Epoch 2/50
757/757 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - loss: 0.3512 - val_loss: 0.1998 - learning_rate: 0.0010
Epoch 3/50
757/757 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - loss: 0.2999 - val_loss: 0.1810 - learning_rate: 0.0010
Epoch 4/50
757/757 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - loss: 0.2627 - val_loss: 0.1746 - learning_rate: 0.0010
Epoch 5/50
757/757 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - loss: 0.2447 - val_loss: 0.1756 - learning_rate: 0.0010
Epoch 6/50
757/757 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - loss: 0.2387 - val_loss: 0.1598 - learning_rate: 0.0010
Epoch 7/50
757/757 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - loss: 0.2327 - val_loss: 0.1624 - learning_rate: 0.0010
Epoch 8/50
757/757 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - loss: 0.2282 - val_loss: 0.1672 - learning_rate: 0.0010
Epoch 9/50
757/757 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - loss: 0.2235 - val_loss: 0.1650 - learning_rate:

In [ ]:
# Generate predictions for the CSV
y_pred = model_power.predict(test_ds_tf)
y_true = np.concatenate([y for x, y in test_ds_tf], axis=0)

# SAVE EVERYTHING
save_model_results('VanillaRNN_power_tf', history, y_true, y_pred)

In [ ]:
# Load weights
model_power.load_weights('best_power_RNN..weights.h5')

# Predict and Flatten
y_pred = model_power.predict(test_ds_tf).flatten()

# Get true values from TF dataset
y_true = np.concatenate([y for x, y in test_ds_tf], axis=0)

print("\n--- Improved RNN Metrics (Electric Power) ---")
print(f"R² Score: {r2_score(y_true, y_pred):.4f}")
print(f"MSE:       {mean_squared_error(y_true, y_pred):.4f}")
print(f"RMSE:      {np.sqrt(mean_squared_error(y_true, y_pred)):.4f}")
print(f"MAE:       {mean_absolute_error(y_true, y_pred):.4f}")

325/325 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step

--- Improved GRU Metrics (Electric Power) ---
R² Score: 0.5241
MSE:       0.3184
RMSE:      0.5642
MAE:       0.3802


2026-03-01 14:21:48.829588: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


## Technique 2: RestNet

In [ ]:

class RegularizedResBlockTF(layers.Layer):
    def __init__(self, out_channels, **kwargs):
        super(RegularizedResBlockTF, self).__init__(**kwargs)
        self.out_channels = out_channels
        self.conv1 = layers.Conv1D(out_channels, kernel_size=3, padding='same')
        self.bn1 = layers.BatchNormalization()
        self.relu = layers.LeakyReLU(alpha=0.1)
        self.dropout = layers.Dropout(0.3)
        self.conv2 = layers.Conv1D(out_channels, kernel_size=3, padding='same')
        self.bn2 = layers.BatchNormalization()
        
    def build(self, input_shape):
        # input_shape is (Batch, Seq, Channels)
        in_channels = input_shape[-1]
        if in_channels != self.out_channels:
            self.shortcut = Sequential([
                layers.Conv1D(self.out_channels, kernel_size=1),
                layers.BatchNormalization()
            ])
        else:
            self.shortcut = lambda x, training=False: x
        super(RegularizedResBlockTF, self).build(input_shape)

    def call(self, x, training=False):
        residual = self.shortcut(x, training=training)
        
        out = self.conv1(x)
        out = self.bn1(out, training=training)
        out = self.relu(out)
        out = self.dropout(out, training=training)
        
        out = self.conv2(out)
        out = self.bn2(out, training=training)
        
        return self.relu(out + residual)

class FinalResNet1DTF(Model):
    def __init__(self, **kwargs):
        super(FinalResNet1DTF, self).__init__(**kwargs)
        self.start = layers.Conv1D(64, kernel_size=3, padding='same')
        self.block1 = RegularizedResBlockTF(128)
        self.block2 = RegularizedResBlockTF(128)
        self.flatten = layers.Flatten()
        self.fc = Sequential([
            layers.Dense(64, activation='relu'),
            layers.Dropout(0.4),
            layers.Dense(1)
        ])

    def call(self, x, training=False):
        # No transpose needed: TF expects (Batch, Seq, Features)
        x = self.start(x)
        x = tf.nn.relu(x)
        x = self.block1(x, training=training)
        x = self.block2(x, training=training)
        x = self.flatten(x)
        return self.fc(x, training=training)

In [51]:
# Setup
model_final = FinalResNet1DTF()
optimizer = tf.keras.optimizers.AdamW(
    learning_rate=0.0005, 
    weight_decay=0.01, 
    global_clipnorm=1.0
)

model_final.compile(optimizer=optimizer, loss=tf.keras.losses.Huber())

final_callbacks = [
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', patience=3, factor=0.5, verbose=1),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(filepath='best_power_resnet.weights.h5', save_best_only=True, save_weights_only=True)
]

# Train
history = model_final.fit(
    train_ds_tf,
    epochs=50,
    validation_data=test_ds_tf,
    callbacks=final_callbacks
)

Epoch 1/50
757/757 ━━━━━━━━━━━━━━━━━━━━ 27s 31ms/step - loss: 4.7751 - val_loss: 0.3217 - learning_rate: 5.0000e-04
Epoch 2/50
757/757 ━━━━━━━━━━━━━━━━━━━━ 23s 31ms/step - loss: 0.7742 - val_loss: 0.2183 - learning_rate: 5.0000e-04
Epoch 3/50
757/757 ━━━━━━━━━━━━━━━━━━━━ 23s 31ms/step - loss: 0.3419 - val_loss: 0.2119 - learning_rate: 5.0000e-04
Epoch 4/50
757/757 ━━━━━━━━━━━━━━━━━━━━ 22s 30ms/step - loss: 0.3369 - val_loss: 0.2144 - learning_rate: 5.0000e-04
Epoch 5/50
757/757 ━━━━━━━━━━━━━━━━━━━━ 22s 30ms/step - loss: 0.3335 - val_loss: 0.2145 - learning_rate: 5.0000e-04
Epoch 6/50
757/757 ━━━━━━━━━━━━━━━━━━━━ 23s 30ms/step - loss: 0.3025 - val_loss: 0.2102 - learning_rate: 5.0000e-04
Epoch 7/50
757/757 ━━━━━━━━━━━━━━━━━━━━ 23s 30ms/step - loss: 0.2779 - val_loss: 0.1871 - learning_rate: 5.0000e-04
Epoch 8/50
757/757 ━━━━━━━━━━━━━━━━━━━━ 23s 30ms/step - loss: 0.2570 - val_loss: 0.2343 - learning_rate: 5.0000e-04
Epoch 9/50
757/757 ━━━━━━━━━━━━━━━━━━━━ 23s 30ms/step - loss: 0.2468 - v

In [ ]:
# Generate predictions for the CSV
y_pred = model_final.predict(test_ds_tf)
y_true = np.concatenate([y for x, y in test_ds_tf], axis=0)

# SAVE EVERYTHING
save_model_results('RestNet_power_tf', history, y_true, y_pred)

In [52]:
model_final.load_weights('best_power_resnet.weights.h5')

y_pred = model_final.predict(test_ds_tf).flatten()
y_true = np.concatenate([y for x, y in test_ds_tf], axis=0)

print(f"\n--- Final Stabilized ResNet Metrics ---")
print(f"R² Score: {r2_score(y_true, y_pred):.4f}")
print(f"MSE:       {mean_squared_error(y_true, y_pred):.4f}")
print(f"MAE:       {mean_absolute_error(y_true, y_pred):.4f}")

325/325 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step

--- Final Stabilized ResNet Metrics ---
R² Score: 0.3882
MSE:       0.4093
MAE:       0.4748


2026-03-01 14:30:20.014187: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
